In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import ast
from sklearn.metrics import accuracy_score
import copy
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score,average_precision_score)
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from scipy.stats import spearmanr
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance

In [2]:
path = "D:\\MissTiny\\GitHub"

In [13]:
clip_embedding = np.load(f"{path}\\Creativity_Artnet\\Datasets\\clip_embedding_2024.npy",allow_pickle=True)

In [14]:
gold_set_70 = pd.read_excel(f"{path}\\Creativity_Artnet\\Codes\\Evaluation\\golden_set_move_creative_70.xlsx")
creative_70 = np.load(f"{path}\\Creativity_Artnet\\Datasets\\creativity_70_full_prob.npy",allow_pickle=True)
artnet_2024 = pd.read_excel(f"{path}\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")
merged_df = pd.merge(gold_set_70, artnet_2024[["artwork id","artist id","year born","workyear modifier","price","artist",'est lo usd','est hi usd', 'sale price usd']], on='artwork id', how='left')

In [15]:
artist_list = ["Pablo Picasso","Pierre-Auguste Renoir","Käthe Kollwitz"]
keep_list = merged_df["artist"].isin(artist_list)
merged_df = merged_df[keep_list]
creative_70 = creative_70[keep_list]

In [27]:
creativity_min = np.min(creative_70)
creativity_max =np.max(creative_70)
creativity_normalized = (creative_70 - creativity_min)/(creativity_max-creativity_min)

In [28]:
embedding_df = np.zeros(merged_df.shape[0], dtype=object)
for i in range(merged_df.shape[0]):
    row_index = merged_df.index[i]
    embedding_df[i] = clip_embedding[row_index]

In [23]:
def logit(p, epsilon=1e-10):
    # Clip p to avoid log(0) or log(1/0)
    p = np.clip(p, epsilon, 1 - epsilon)
    return np.log(p / (1 - p))

In [24]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [25]:
def topk_hit_rate_curve(y_true, y_pred, rates):
    hit_rates = []
    n = len(y_true)

    for rate in rates:
        k = int(n * rate)
        if k == 0:
            hit_rates.append(np.nan)
            continue

        true_top_idx = np.argsort(y_true)[-k:]
        pred_top_idx = np.argsort(y_pred)[-k:]

        hit_rate = len(set(true_top_idx) & set(pred_top_idx)) / k
        hit_rates.append(hit_rate)

    return hit_rates

In [26]:
def kl_divergence(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    # Normalize (important if they are not exact probabilities)
    p = p / p.sum()
    q = q / q.sum()

    # Avoid log(0)
    p = np.clip(p, eps, 1)
    q = np.clip(q, eps, 1)

    return np.sum(p * np.log(p / q))

# Neural Network

In [30]:
epsilon=1e-6

In [31]:
y = np.log((creativity_normalized+epsilon).astype(float))

In [33]:
y.shape

(1862,)

In [37]:
X = np.stack(embedding_df)

In [43]:
X.shape

(1862, 512)

In [44]:
alphas = [1e-6,3e-6,5e-6,7e-6,9e-6,1e-5,1e-4,1e-3,1e-2]
layers=[(128,),(256,),(128, 64),(256, 128),(256, 128, 64)]

In [45]:
result=[]
best_alpha=None
best_layer=None
best_score=None
for alpha in alphas:
    for layer in layers:
        model = MLPRegressor(solver='lbfgs', alpha=alpha,
                    hidden_layer_sizes=layer, random_state=1,max_iter=5000)
        X_train1, X_val, y_train1, y_val = train_test_split(X,y, test_size=0.20, random_state=23)
        model.fit(X_train1, y_train1)
        
        true_label = sigmoid(y_val)
        pred_y = sigmoid(model.predict(X_val)) 

        mse = mean_squared_error(true_label, np.array(pred_y))
        #print(f"Test Linear Regression model mse: {mse:.4f}")
        mae = mean_absolute_error(true_label, pred_y)
        #print("Test mae:", mae)
        RMSE = root_mean_squared_error(true_label, pred_y)
        #print("Test RMSE:", RMSE)
        #rho, pval = spearmanr(true_label, pred_y)
        #print("Test spearman:", rho)

        kl = kl_divergence(true_label, pred_y)
        # print("Test KL:", kl )
        js = jensenshannon(true_label, pred_y)**2
        # print("Test JS:", js )
        wd = wasserstein_distance(true_label, pred_y)
        
        if best_alpha==None:
            best_alpha = alpha
            best_layer = layer
            best_score = RMSE
        elif best_score < RMSE:
            best_alpha = alpha
            best_layer = layer
            best_score = RMSE
        
        print(f"alpha: {alpha}, layer: {layer}, mse: {mse}, mae:{mae}, RMSE: {RMSE}, kl: {kl}, js: {js}, wd: {wd}")
        result.append({'alpha':alpha,'layer':layer,'mse':mse,'mae':mae,'RMSE':RMSE,'kl':kl,'js':js, 'wd': wd})

alpha: 1e-06, layer: (128,), mse: 0.003971811366429968, mae:0.04089711360492337, RMSE: 0.06302230848223483, kl: 0.011917098728932003, js: 0.003411988735035051, wd: 0.01424489690439665
alpha: 1e-06, layer: (256,), mse: 0.004179382886241814, mae:0.042891648170845315, RMSE: 0.06464814681212304, kl: 0.012563578722408337, js: 0.003564357892139479, wd: 0.013362098035695396
alpha: 1e-06, layer: (128, 64), mse: 0.00397810054612062, mae:0.040874616308435634, RMSE: 0.06307218520172438, kl: 0.012084805161342367, js: 0.003441545270747096, wd: 0.013409790774594355
alpha: 1e-06, layer: (256, 128), mse: 0.004230744532615397, mae:0.04241457068992577, RMSE: 0.06504417370230324, kl: 0.01280169416420129, js: 0.003631364955462949, wd: 0.013530217204004095
alpha: 1e-06, layer: (256, 128, 64), mse: 0.004136600261412966, mae:0.03942622455569857, RMSE: 0.0643164074044327, kl: 0.012752478627185228, js: 0.0036218944112647606, wd: 0.011174101302203154
alpha: 3e-06, layer: (128,), mse: 0.004170689973105445, mae:0

C:\Users\MissTiny\anaconda3\envs\Creativity\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


# Pretrained Model + Classifier